# arXiv Computer Science 최근 3개월 논문 수집 (OAI-PMH)

arXiv OAI-PMH의 `ListRecords`로 Computer Science(`cs`) 메타데이터를 수집한 뒤, 원하는 6개 세부 카테고리만 로컬에서 필터링하여 JSONL로 저장합니다.

OAI-PMH는 대량 메타데이터 수집에 적합하며 `resumptionToken`을 사용해 페이지를 이어받습니다. 저장 위치는 프로젝트 루트의 `data/ai/`입니다.

In [1]:
from calendar import monthrange
from datetime import date
import json
from pathlib import Path
import ssl
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

OAI_URL = 'https://export.arxiv.org/oai2'
METADATA_PREFIX = 'arXiv'
OAI_SET = 'cs'
TARGET_CATEGORIES = {'cs.AI', 'cs.LG', 'cs.CL', 'cs.CV', 'cs.IR', 'cs.RO'}
REQUEST_INTERVAL_SECONDS = 3.0
MAX_RETRIES = 8
BACKOFF_BASE_SECONDS = 10.0
BACKOFF_MAX_SECONDS = 300.0
CHUNK_SIZE = 5000
FILE_PREFIX = 'arxiv_cs_recent_3months_oai'

def subtract_months(day, months):
    month_index = day.year * 12 + day.month - 1 - months
    year, month_zero_based = divmod(month_index, 12)
    month = month_zero_based + 1
    return date(year, month, min(day.day, monthrange(year, month)[1]))

END_DATE = date.today()
START_DATE = subtract_months(END_DATE, 3)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'ai'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = OUTPUT_DIR / f'{FILE_PREFIX}_state.json'

def chunk_path(index):
    return OUTPUT_DIR / f'{FILE_PREFIX}_part{index}.jsonl'

print(f'조회 기간: {START_DATE} ~ {END_DATE}')
print(f'대상 카테고리: {sorted(TARGET_CATEGORIES)}')
print(f'저장 위치: {OUTPUT_DIR.resolve()}')

조회 기간: 2026-06-14 ~ 2026-09-14
대상 카테고리: ['cs.AI', 'cs.CL', 'cs.CV', 'cs.IR', 'cs.LG', 'cs.RO']
저장 위치: C:\Users\oh\Desktop\arxiv_graph_RAG\data\ai


In [2]:
ATOM = 'http://www.w3.org/2005/Atom'
OAI = 'http://www.openarchives.org/OAI/2.0/'
ARXIV = 'http://arxiv.org/OAI/arXiv/'
NS = {'atom': ATOM, 'oai': OAI, 'arxiv': ARXIV}
_last_request_at = 0.0

def _text(node, path):
    child = node.find(path, NS)
    return child.text.strip() if child is not None and child.text else None

def _retry_delay(error, attempt):
    retry_after = error.headers.get('Retry-After') if getattr(error, 'headers', None) else None
    try:
        return max(float(retry_after), REQUEST_INTERVAL_SECONDS) if retry_after else min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)
    except (TypeError, ValueError):
        return min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)

def _get_xml(params):
    global _last_request_at
    url = f'{OAI_URL}?{urllib.parse.urlencode(params)}'
    request = urllib.request.Request(url, headers={'User-Agent': 'arxiv-cs-oai-harvester/1.0'})
    for attempt in range(MAX_RETRIES + 1):
        elapsed = time.monotonic() - _last_request_at
        if elapsed < REQUEST_INTERVAL_SECONDS:
            time.sleep(REQUEST_INTERVAL_SECONDS - elapsed)
        try:
            _last_request_at = time.monotonic()
            with urllib.request.urlopen(request, timeout=120, context=ssl.create_default_context()) as response:
                return ET.fromstring(response.read())
        except urllib.error.HTTPError as error:
            if error.code not in {429, 500, 502, 503, 504} or attempt >= MAX_RETRIES:
                raise
            delay = _retry_delay(error, attempt)
            print(f'HTTP {error.code}: {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            error.close()
            time.sleep(delay)

def parse_record(record):
    header = record.find('oai:header', NS)
    if header is None or header.attrib.get('status') == 'deleted':
        return None
    metadata = record.find('oai:metadata/arxiv:arXiv', NS)
    if metadata is None:
        return None
    categories = (_text(metadata, 'arxiv:categories') or '').split()
    created = _text(metadata, 'arxiv:created')
    if not created or not (START_DATE.isoformat() <= created <= END_DATE.isoformat()):
        return None
    if not TARGET_CATEGORIES.intersection(categories):
        return None
    authors = [_text(author, 'arxiv:forenames') or '' for author in metadata.findall('arxiv:authors/arxiv:author', NS)]
    for index, author in enumerate(metadata.findall('arxiv:authors/arxiv:author', NS)):
        keyname = _text(author, 'arxiv:keyname') or ''
        authors[index] = f'{authors[index]} {keyname}'.strip()
    identifier = _text(metadata, 'arxiv:id') or _text(header, 'oai:identifier')
    updated = _text(metadata, 'arxiv:updated') or created
    primary_node = metadata.find('arxiv:primary_category', NS)
    return {
        'id': f'https://arxiv.org/abs/{identifier}' if identifier and not identifier.startswith('http') else identifier,
        'title': ' '.join((_text(metadata, 'arxiv:title') or '').split()),
        'abstract': ' '.join((_text(metadata, 'arxiv:abstract') or '').split()),
        'authors': [name for name in authors if name],
        'categories': categories,
        'primary_category': primary_node.attrib.get('term') if primary_node is not None else None,
        'published': f'{created}T00:00:00Z' if created else None,
        'updated': f'{updated}T00:00:00Z' if updated else None,
        'doi': _text(metadata, 'arxiv:doi'),
        'pdf_url': f'https://arxiv.org/pdf/{identifier}' if identifier else None,
        'source': 'arxiv',
        'collection_window': {'start': START_DATE.isoformat(), 'end': END_DATE.isoformat()},
    }

def fetch_page(resumption_token=None):
    if resumption_token:
        root = _get_xml({'verb': 'ListRecords', 'resumptionToken': resumption_token})
    else:
        root = _get_xml({'verb': 'ListRecords', 'from': START_DATE.isoformat(), 'until': END_DATE.isoformat(), 'set': OAI_SET, 'metadataPrefix': METADATA_PREFIX})
    error = root.find('oai:error', NS)
    if error is not None:
        raise RuntimeError(f"OAI-PMH {error.attrib.get('code')}: {error.text}")
    records = [paper for paper in (parse_record(record) for record in root.findall('oai:ListRecords/oai:record', NS)) if paper]
    token_node = root.find('oai:ListRecords/oai:resumptionToken', NS)
    next_token = token_node.text.strip() if token_node is not None and token_node.text else None
    return records, next_token

In [3]:
def load_existing():
    papers = []
    index = 1
    while chunk_path(index).exists():
        with chunk_path(index).open(encoding='utf-8') as file:
            papers.extend(json.loads(line) for line in file if line.strip())
        index += 1
    return papers

def write_chunks(papers):
    for index in range(1, max(1, (len(papers) + CHUNK_SIZE - 1) // CHUNK_SIZE) + 1):
        start = (index - 1) * CHUNK_SIZE
        with chunk_path(index).open('w', encoding='utf-8') as file:
            for paper in papers[start:start + CHUNK_SIZE]:
                file.write(json.dumps(paper, ensure_ascii=False) + '\n')

state = json.loads(STATE_PATH.read_text(encoding='utf-8')) if STATE_PATH.exists() else None
query_key = {'start': START_DATE.isoformat(), 'end': END_DATE.isoformat(), 'set': OAI_SET, 'metadata_prefix': METADATA_PREFIX}
papers = load_existing() if state and state.get('query') == query_key else []
seen_ids = {paper['id'] for paper in papers if paper.get('id')}
token = state.get('resumption_token') if state and state.get('query') == query_key and papers else None
seen_tokens = set()
if not papers and state and state.get('query') != query_key:
    for old_file in OUTPUT_DIR.glob(f'{FILE_PREFIX}_part*.jsonl'):
        old_file.unlink()
    if STATE_PATH.exists():
        STATE_PATH.unlink()

while True:
    if token and token in seen_tokens:
        raise RuntimeError('동일한 resumptionToken이 반복되어 수집을 중단했습니다.')
    if token:
        seen_tokens.add(token)
    page, token = fetch_page(token)
    for paper in page:
        if paper['id'] and paper['id'] not in seen_ids:
            seen_ids.add(paper['id'])
            papers.append(paper)
    write_chunks(papers)
    STATE_PATH.write_text(json.dumps({'query': query_key, 'resumption_token': token}, ensure_ascii=False), encoding='utf-8')
    print(f"누적 수집: {len(papers)}건; 다음 페이지 토큰: {'있음' if token else '없음'}")
    if not token:
        STATE_PATH.unlink(missing_ok=True)
        break

print(f'완료: {len(papers)}건을 {max(1, (len(papers) + CHUNK_SIZE - 1) // CHUNK_SIZE)}개 JSONL 파일로 저장했습니다.')
assert all(START_DATE.isoformat() <= paper['published'][:10] <= END_DATE.isoformat() for paper in papers)
assert all(TARGET_CATEGORIES.intersection(paper['categories']) for paper in papers)
assert len({paper['id'] for paper in papers}) == len(papers)
print('검증 완료: 날짜 범위, 카테고리, 중복 조건을 통과했습니다.')

누적 수집: 0건; 다음 페이지 토큰: 있음
누적 수집: 887건; 다음 페이지 토큰: 있음
누적 수집: 1766건; 다음 페이지 토큰: 있음
누적 수집: 2620건; 다음 페이지 토큰: 있음
누적 수집: 3525건; 다음 페이지 토큰: 있음
누적 수집: 4447건; 다음 페이지 토큰: 있음
누적 수집: 5327건; 다음 페이지 토큰: 있음
누적 수집: 6213건; 다음 페이지 토큰: 있음
누적 수집: 7160건; 다음 페이지 토큰: 있음
누적 수집: 8090건; 다음 페이지 토큰: 있음
누적 수집: 8941건; 다음 페이지 토큰: 있음
누적 수집: 9845건; 다음 페이지 토큰: 있음
누적 수집: 10737건; 다음 페이지 토큰: 있음
누적 수집: 11594건; 다음 페이지 토큰: 있음
누적 수집: 12459건; 다음 페이지 토큰: 있음
누적 수집: 13279건; 다음 페이지 토큰: 있음
누적 수집: 14129건; 다음 페이지 토큰: 있음
누적 수집: 14942건; 다음 페이지 토큰: 있음
누적 수집: 15791건; 다음 페이지 토큰: 있음
누적 수집: 16634건; 다음 페이지 토큰: 있음
누적 수집: 17484건; 다음 페이지 토큰: 있음
누적 수집: 18327건; 다음 페이지 토큰: 있음
누적 수집: 19180건; 다음 페이지 토큰: 있음
누적 수집: 20023건; 다음 페이지 토큰: 있음
누적 수집: 20895건; 다음 페이지 토큰: 있음
누적 수집: 21782건; 다음 페이지 토큰: 있음
누적 수집: 22664건; 다음 페이지 토큰: 있음
누적 수집: 23588건; 다음 페이지 토큰: 있음
누적 수집: 24482건; 다음 페이지 토큰: 있음
누적 수집: 25364건; 다음 페이지 토큰: 있음
누적 수집: 26236건; 다음 페이지 토큰: 있음
누적 수집: 27171건; 다음 페이지 토큰: 있음
누적 수집: 28087건; 다음 페이지 토큰: 있음
누적 수집: 28985건; 다음 페이지 토큰: 있음
누적 수집: 29908건; 다음 페이지 토큰: 있음
누